In [8]:
import pandas as pd
import numpy as np
import duckdb

In [9]:
df_teams = duckdb.query("""
SELECT
    c.*,
    t.rank,
    t.confederation,
    IF(tu.host_country = c.team, 1, 0) as host
FROM (
    SELECT DISTINCT
        *
    FROM (
        SELECT 
            tournament_id,
            home_team_name as team
        FROM '../data/db/matches.csv'
        WHERE 
            tournament_id >= 'WC-2022'
        UNION ALL
        SELECT 
            tournament_id,
            away_team_name as team
        FROM '../data/db/matches.csv'
        WHERE 
            tournament_id >= 'WC-1998'
    )
    -- WHERE tournament_id = 'WC-2022' AND team = 'Argentina'
) as c
INNER JOIN '../data/db/tournaments.csv' as tu ON tu.tournament_id = c.tournament_id AND tu.tournament_name LIKE '%Men%'
LEFT JOIN '../data/wc_teams.csv' as t 
    ON t.team = c.team AND CONCAT('WC-', t.year) = c.tournament_id
""").df()
df_teams

,tournament_id,team,rank,confederation,host
0,WC-1998,Iran,42,AFC,0
1,WC-1998,Japan,12,AFC,0
2,WC-1998,Saudi Arabia,34,AFC,0
3,WC-1998,South Korea,20,AFC,0
4,WC-1998,Cameroon,49,CAF,0
...,...,...,...,...,...
214,WC-2022,Portugal,9,UEFA,0
215,WC-2022,Serbia,21,UEFA,0
216,WC-2022,Spain,7,UEFA,0
217,WC-2022,Switzerland,15,UEFA,0


In [10]:
df_teams[df_teams['rank'].isnull()], df_teams.host.sum()


(Empty DataFrame
 Columns: [tournament_id, team, rank, confederation, host]
 Index: [],
 np.int64(6))

In [11]:
duckdb.query("""
CREATE OR REPLACE TEMP MACRO get_opp_name(team, home_team_name, away_team_name) AS
    CASE 
        WHEN team = home_team_name THEN away_team_name
        ELSE home_team_name
    END
;

CREATE OR REPLACE TEMP MACRO get_value(team, home_team_name, away_team_name, home_val, away_val) AS
    CASE 
        WHEN team = home_team_name THEN home_val
        ELSE away_val
    END
;
""") 

In [12]:
df = duckdb.query("""
WITH 
    team_stats as (
    SELECT
        tournament_id, match_id, stage_name, match_date,

        team, confederation, rank, host,
        matches_played, lst_match_days_ago, 

        /*
        ARRAY_AGG(score) OVER (PARTITION BY tournament_id, team 
                        ORDER BY match_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as goals_sco_hist,
        ARRAY_AGG(opp_score) OVER (PARTITION BY tournament_id, team 
                        ORDER BY match_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as goals_con_hist,
        ARRAY_AGG(win) OVER (PARTITION BY tournament_id, team 
                        ORDER BY match_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as win_hist,
        ARRAY_AGG(draw) OVER (PARTITION BY tournament_id, team 
                        ORDER BY match_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as draw_hist,
        ARRAY_AGG(opp_team_rank) OVER (PARTITION BY tournament_id, team 
                        ORDER BY match_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) as opp_team_rank_hist,
        */

        score, win, draw
    FROM (
        SELECT 
            m.tournament_id, m.match_id, m.stage_name, m.match_date,

            t.team,
            t.confederation,
            t.rank,
            t.host,

            COUNT(*) OVER (PARTITION BY m.tournament_id, t.team ORDER BY m.match_date) as matches_played,
            IFNULL(m.match_date - LAG(m.match_date) OVER (PARTITION BY m.tournament_id, t.team ORDER BY m.match_date), 0) as lst_match_days_ago,
            get_value(t.team, m.home_team_name, m.away_team_name, home_team_score, away_team_score) as score,
            get_value(get_opp_name(t.team, m.home_team_name, m.away_team_name), m.home_team_name, m.away_team_name, home_team_score, away_team_score) as opp_score,
            get_value(t.team, m.home_team_name, m.away_team_name, home_team_win, away_team_win) as win,
            m.draw,

            (SELECT rank FROM df_teams as tm 
            WHERE tm.tournament_id = m.tournament_id AND tm.team = get_opp_name(t.team, m.home_team_name, m.away_team_name)
            LIMIT 1) as opp_team_rank


        FROM '../data/db/matches.csv' as m
        INNER JOIN df_teams as t
            ON t.tournament_id = m.tournament_id
                AND (t.team = m.home_team_name OR t.team = m.away_team_name)
    )
)

SELECT 
    ts.*, --EXCLUDE(tournament_id, match_id, match_date), 
    ts_opp.team as opp_team,
    ts_opp.confederation as opp_confederation,
    ts_opp.rank as opp_rank,
    ts_opp.host as opp_host,
    ts_opp.lst_match_days_ago as opp_lst_match_days_ago

FROM team_stats as ts
INNER JOIN team_stats as ts_opp
    ON ts.tournament_id = ts_opp.tournament_id
        AND ts.match_id = ts_opp.match_id
        AND ts.team != ts_opp.team
-- WHERE ts.tournament_id = 'WC-2022' AND ts.team = 'Argentina'
ORDER BY ts.match_date ASC
""").df()   
df

,tournament_id,match_id,stage_name,match_date,team,confederation,rank,host,matches_played,lst_match_days_ago,score,win,draw,opp_team,opp_confederation,opp_rank,opp_host,opp_lst_match_days_ago
0,WC-1998,M-1998-02,group stage,1998-06-10,Morocco,CAF,13,0,1,0,2,0,1,Norway,UEFA,7,0,0
1,WC-1998,M-1998-02,group stage,1998-06-10,Norway,UEFA,7,0,1,0,2,0,1,Morocco,CAF,13,0,0
2,WC-1998,M-1998-04,group stage,1998-06-11,Austria,UEFA,31,0,1,0,1,0,1,Cameroon,CAF,49,0,0
3,WC-1998,M-1998-04,group stage,1998-06-11,Cameroon,CAF,49,0,1,0,1,0,1,Austria,UEFA,31,0,0
4,WC-1998,M-1998-05,group stage,1998-06-12,Paraguay,CONMEBOL,29,0,1,0,0,0,1,Bulgaria,UEFA,35,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843,WC-2022,M-2022-62,semi-finals,2022-12-14,France,UEFA,4,0,6,4,2,1,0,Morocco,CAF,22,0,4
844,WC-2022,M-2022-63,third-place match,2022-12-17,Morocco,CAF,22,0,7,3,1,0,0,Croatia,UEFA,12,0,4
845,WC-2022,M-2022-63,third-place match,2022-12-17,Croatia,UEFA,12,0,7,4,2,1,0,Morocco,CAF,22,0,3
846,WC-2022,M-2022-64,final,2022-12-18,Argentina,CONMEBOL,3,0,7,5,3,1,0,France,UEFA,4,0,4


In [13]:
df.to_json("../data/dataset.json")